# Глава 6. Численные эксперименты для объёмной электродинамической постановки

Notebook использует модульный CPU-first harness `experiments.chapter06_em`. Быстрый режим держит расчёты лёгкими, а дорогие эксперименты `N=64` включаются флагом `RUN_FULL`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

import em3d
from experiments import chapter06_em as c6

RUN_MODE = "quick"
RUN_FULL = True
RUN_CRASH_TEST = True
OUTPUT_ROOT = PROJECT_ROOT / "experiments" / "outputs" / "chapter06"
paths = c6.ensure_output_dirs(OUTPUT_ROOT)
LOGGER = c6.ExperimentLogger(OUTPUT_ROOT, "chapter06-notebook")
LOGGER.event("start", notebook="chapter-06-em", run_mode=RUN_MODE, run_full=RUN_FULL)
N_VALUES = c6.n_series_for_mode(RUN_MODE)
N_VALUES


## 6.1. Обоснование использования электродинамической постановки

В экспериментальном блоке задаются изотропные, анизотропные, анизотропные поглощающие и Drude-плазменные материалы. Все они приводятся к формату `(eps_real, eps_imag)`, совместимому с векторным оператором.

In [ ]:
materials = [
    c6.MaterialSpec.isotropic(2.0),
    c6.MaterialSpec.anisotropic(np.diag([2.0, 1.6, 1.3])),
    c6.MaterialSpec.anisotropic_lossy(
        np.diag([2.0, 1.6, 1.3]),
        np.diag([0.05, 0.03, 0.02]),
    ),
    c6.MaterialSpec.plasma_drude(eps_inf=1.0, omega_p=2.0, gamma=0.1),
]
[(material.kind, c6.material_eps(material, k0=2.0)) for material in materials]


## 6.2. Постановка вычислительных экспериментов для анизотропных диэлектрических структур

Одноосный кристалл моделируется эллипсоидом с `eps = diag(eps_o, eps_o, eps_e)`. Анизотропия проявляется в различии карт `|E|`, `|Ex|`, `|Ey|`, `|Ez|` в сечениях `xy`, `xz`, `yz`.

In [ ]:
crystal_case = c6.make_uniaxial_crystal_ellipsoid_case(
    N=32, eps_o=2.2, eps_e=1.4, k0=3.0
)
crystal_problem, crystal_operator = c6.build_problem(crystal_case)
crystal_result = em3d.BiCGStab(em3d.SolverConfig(max_iter=500, rtol=1e-6)).solve(
    crystal_operator, crystal_problem.wave
)
crystal_u = np.asarray(crystal_result.u)
c6.plot_three_field_slices(crystal_u, crystal_problem.grid, part="abs")


## 6.3. Дискретный спектр оператора и его влияние на сходимость стационарных итераций

Оценка `gamma0` проводится для трёх сценариев: изотропный эллипсоид, анизотропный эллипсоид и трёхслойный прямоугольный параллелепипед с растущим комплексным анизотропным контрастом.

In [ ]:
coarse_values = [2, 3, 4, 5, 6]
k_values = np.arange(1, 11)
gamma0_iso = c6.scan_gamma0(
    c6.make_isotropic_gamma0_case,
    coarse_values=coarse_values,
    k_values=k_values,
    scenario="isotropic-ellipsoid",
)
gamma0_aniso = c6.scan_gamma0(
    c6.make_anisotropic_gamma0_case,
    coarse_values=coarse_values,
    k_values=k_values,
    scenario="anisotropic-ellipsoid",
)
gamma0_layered = c6.scan_gamma0(
    c6.make_layered_gamma0_case,
    coarse_values=coarse_values,
    k_values=k_values,
    scenario="layered-box",
)
gamma0_iso[:2], gamma0_aniso[:2], gamma0_layered[:2]


## 6.4. Применение обобщённого метода простой итерации

Для `N=64`, `k0=10` и анизотропного эллипсоида сравнивается SIM при разных грубых сетках оценки `gamma0`: `2..6`.

In [ ]:
if RUN_FULL:
    n64_case = c6.make_anisotropic_ellipsoid_case(
        N=64,
        eps_real=np.diag([2.0, 1.6, 1.3]),
        eps_imag=np.zeros((3, 3)),
        k0=10.0,
    )
    sim_scan = c6.scan_sim_convergence_by_gamma0(
        n64_case, coarse_values=[2, 3, 4, 5, 6], max_iter=500, rtol=1e-6
    )
else:
    n64_case = c6.make_anisotropic_ellipsoid_case(
        N=8,
        eps_real=np.diag([2.0, 1.6, 1.3]),
        eps_imag=np.zeros((3, 3)),
        k0=10.0,
    )
    sim_scan = []
sim_scan[:1]


## 6.5. Применение BiCGStab и сравнение с авторскими модификациями итерационных методов

Сравниваются SIM, BiCGStab и TwoStep. Для SIM используется `gamma0`, оценённый на грубой сетке `N=6`.

In [ ]:
if RUN_FULL:
    comparison = c6.run_solver_comparison(
        n64_case, sim_coarse_N=6, max_iter=500, rtol=1e-6
    )
else:
    comparison = c6.run_solver_comparison(
        c6.make_anisotropic_ellipsoid_case(
            N=8,
            eps_real=np.diag([1.3, 1.2, 1.1]),
            eps_imag=np.zeros((3, 3)),
            k0=1.0,
        ),
        sim_coarse_N=2,
        max_iter=20,
        rtol=1e-4,
    )
[run.to_row() for run in comparison["runs"]]


## 6.6. FFT-ускоренное матрично-векторное умножение для векторного оператора

FFT-backed `matvec` сравнивается с плотным NumPy-умножением только на малых сетках `N=2..10`, потому что dense matrix быстро становится слишком дорогой.

In [ ]:
def dense_benchmark_factory(N):
    return c6.make_anisotropic_ellipsoid_case(
        N=N,
        eps_real=np.diag([1.2, 1.1, 1.05]),
        eps_imag=np.zeros((3, 3)),
        k0=1.0,
    )

fft_dense_rows = c6.benchmark_fft_vs_dense(
    dense_benchmark_factory,
    n_values=c6.FFT_DENSE_N_VALUES,
    repeats=3,
    logger=LOGGER,
)
c6.plot_fft_vs_dense_timing(fft_dense_rows, output_dir=paths["figures"])
fft_dense_rows


## 6.7. Расчёт распределения электрического поля внутри области неоднородности

Для поля строятся scalar-срезы нормы и компонент. `mie_field_at` не используется как near-field oracle.

In [ ]:
reference_u = np.asarray(comparison["reference_u"])
reference_grid = comparison["problem"].grid
c6.plot_three_field_slices(reference_u, reference_grid, part="abs")
for component in (0, 1, 2):
    c6.plot_three_field_slices(reference_u, reference_grid, part="abs", component=component)


## 6.8. Расчёт диаграммы направленности и эффективной поверхности рассеяния

Для изотропной сферы меняется size parameter `k0a`; численная нормированная ЭПР сравнивается с кривой Ми.

In [ ]:
rcs_scan = c6.scan_mie_rcs_by_k0a(
    N=32 if RUN_FULL else 8,
    a=c6.RCS_DEFAULT_RADIUS,
    eps_r=2.0,
    k0a_values=c6.RCS_K0A_VALUES,
    n_phi=90,
    max_iter=500,
    rtol=1e-6,
    logger=LOGGER,
)
c6.plot_rcs_scan(rcs_scan, output_dir=paths["figures"])
[{k: row[k] for k in ("k0a", "shape_err", "scale_ratio")} for row in rcs_scan]


## 6.9. Сравнение вычислительной эффективности методов

Агрегируются таблицы времени `matvec`, числа итераций и финальных невязок. Быстрый smoke-run сохраняет CSV/JSON artifacts.

In [ ]:
summary = c6.run_quick_experiment(
    output_root=OUTPUT_ROOT,
    n_values=N_VALUES[:1],
    solver_names=["BiCGStab"],
    max_iter=100,
    rtol=1e-6,
    rcs_n_phi=90,
    logger=LOGGER,
)
summary


## 6.10. Выводы по главе

Выводы формируются после полного запуска `RUN_FULL=True` и отдельного crash-test блока `RUN_CRASH_TEST=True`: анализируются чувствительность `gamma0`, сходимость SIM/BiCGStab/TwoStep, ускорение FFT-матвека и изменение нормированной ЭПР при росте `k0a`.

In [ ]:
if RUN_CRASH_TEST:
    crash_layers = [
        c6.LayerSpec(
            -0.5,
            -1.0 / 6.0,
            c6.MaterialSpec.anisotropic_lossy(
                np.diag([1.8, 1.5, 1.2]), np.diag([0.03, 0.02, 0.01])
            ),
        ),
        c6.LayerSpec(
            -1.0 / 6.0,
            1.0 / 6.0,
            c6.MaterialSpec.anisotropic_lossy(
                np.diag([2.4, 2.0, 1.6]), np.diag([0.06, 0.04, 0.02])
            ),
        ),
        c6.LayerSpec(
            1.0 / 6.0,
            0.5,
            c6.MaterialSpec.anisotropic_lossy(
                np.diag([3.0, 2.5, 2.0]), np.diag([0.09, 0.06, 0.03])
            ),
        ),
    ]
    crash_case = c6.make_layered_box_case(N=128, k0=10.0, layers=crash_layers)
    crash_logger = c6.ExperimentLogger(OUTPUT_ROOT, "twostep-layered-n128")
    crash_logger.event("start", case_name=crash_case.name, solver_name="TwoStep")
    crash_problem, crash_operator = c6.build_problem(crash_case)
    crash_result = em3d.TwoStep(em3d.SolverConfig(max_iter=500, rtol=1e-6)).solve(
        crash_operator, crash_problem.wave
    )
    crash_logger.event(
        "solver_finished",
        case_name=crash_case.name,
        solver_name="TwoStep",
        converged=bool(crash_result.converged),
        iterations=int(crash_result.iterations),
    )
    crash_u = np.asarray(crash_result.u)
    c6.plot_three_field_slices(
        crash_u, crash_problem.grid, part="abs", output_dir=paths["figures"], prefix="twostep_n128"
    )
    for plane in ("xy", "xz", "yz"):
        em3d.vis.plot_field_vector_slice(
            crash_u,
            crash_problem.grid,
            plane=plane,
            part="real",
            filename=str(paths["figures"] / f"twostep_n128_vector_{plane}.png"),
        )
    c6.plot_residual_histories(
        [{"solver_name": "TwoStep", "residual_history": crash_result.residual_history}],
        output_dir=paths["figures"],
        filename="twostep_n128_residual.png",
    )
else:
    "RUN_CRASH_TEST=False; включите флаг для CPU crash-test N=128."
